# Tech Tree Usability Testing — Data Cleaning
This notebook reads the raw Google Form CSV, cleans it, and exports a tidy CSV for the Streamlit dashboard.

In [1]:
import pandas as pd

df = pd.read_csv('../Tech Tree Usability Testing (Responses) - Form Responses 1.csv')
df.head()

,Timestamp,Name (First and Last),Age,Briefly describe your background and who you are,"On a scale of 1 to 10, how was your experience with the interface? (1 = not user-friendly at all, 10 = super user friendly)","When you landed on the homepage, did you immediately understand what the site was about? If not, what was unclear?",Were you able to figure out how to expand or explore nodes in the graph without instructions?,"Did you notice any difficulty distinguishing between node types (e.g., companies, technologies, concepts)?","Did you find yourself wanting a “back” or “reset” function while navigating the graph? If so, at what point?","Were the visual cues (colors, shapes, sizes) helpful in conveying relationships or categories?",Would you recommend this to a friend?,How would you describe your experience with this product to a friend?,Is there anything you would like us to know?
0,7/24/2025 19:40:10,Steve Yang,Over 40,Engineer,7,Yes,Yes,No,NaN,Kinda,Yes,Not bad,No
1,7/24/2025 19:45:20,Wendy Ma,Over 40,Middle age career woman,8,Pretty clear.,Yes,No,No,Yes,Yes,Interesting tool to understand the knowledge o...,Probably it is helpful to have the inf. box mo...
2,7/24/2025 21:12:54,Katie ou,Over 40,"I’m a freelance food and health planner, helpi...",8,Yes,NaN,No,Yes,NaN,Maybe,NaN,NaN
3,7/25/2025 8:43:08,Hannah Yu,Over 40,NaN,1,Yes,No,No,No,yes,Maybe,NaN,No
4,7/25/2025 9:18:44,John Guo,31-40,Biostatistician working on Parkinson's Disease...,3,Yes,"Not sure what you meant by ""expanding"". When I...","Yes, I didn't see any companies in the nodes.",No.,"No. when a node was selected, the size/positio...",No,"Great idea, has a lot of potential, needs a lo...",Steam engine overlapping with vaccination - tw...


In [2]:
# Rename columns to short, code-friendly names
df.columns = [
    'timestamp',
    'name',
    'age_group',
    'background',
    'experience_rating',
    'understood_homepage',
    'explore_without_instructions',
    'difficulty_node_types',
    'wanted_back_reset',
    'visual_cues_helpful',
    'would_recommend',
    'describe_experience',
    'additional_feedback'
]
df.head()

,timestamp,name,age_group,background,experience_rating,understood_homepage,explore_without_instructions,difficulty_node_types,wanted_back_reset,visual_cues_helpful,would_recommend,describe_experience,additional_feedback
0,7/24/2025 19:40:10,Steve Yang,Over 40,Engineer,7,Yes,Yes,No,NaN,Kinda,Yes,Not bad,No
1,7/24/2025 19:45:20,Wendy Ma,Over 40,Middle age career woman,8,Pretty clear.,Yes,No,No,Yes,Yes,Interesting tool to understand the knowledge o...,Probably it is helpful to have the inf. box mo...
2,7/24/2025 21:12:54,Katie ou,Over 40,"I’m a freelance food and health planner, helpi...",8,Yes,NaN,No,Yes,NaN,Maybe,NaN,NaN
3,7/25/2025 8:43:08,Hannah Yu,Over 40,NaN,1,Yes,No,No,No,yes,Maybe,NaN,No
4,7/25/2025 9:18:44,John Guo,31-40,Biostatistician working on Parkinson's Disease...,3,Yes,"Not sure what you meant by ""expanding"". When I...","Yes, I didn't see any companies in the nodes.",No.,"No. when a node was selected, the size/positio...",No,"Great idea, has a lot of potential, needs a lo...",Steam engine overlapping with vaccination - tw...


In [3]:
# Parse timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M:%S')

# Ensure experience_rating is numeric
df['experience_rating'] = pd.to_numeric(df['experience_rating'], errors='coerce')

df.dtypes

timestamp                       datetime64[ns]
name                                    object
age_group                               object
background                              object
experience_rating                        int64
understood_homepage                     object
explore_without_instructions            object
difficulty_node_types                   object
wanted_back_reset                       object
visual_cues_helpful                     object
would_recommend                         object
describe_experience                     object
additional_feedback                     object
dtype: object

In [4]:
def normalize_yes_no(val):
    """Map free-text answers to Yes / No / Unclear."""
    if pd.isna(val):
        return 'Unclear'
    v = str(val).strip().lower()
    if v in ('yes', 'immediately', 'kinda', 'pretty clear.', 'yes.', 'yes '):
        return 'Yes'
    if v in ('no', 'no.', 'not really', 'na', 'n/a'):
        return 'No'
    # Longer answers that start with yes/no
    if v.startswith('yes'):
        return 'Yes'
    if v.startswith('no'):
        return 'No'
    return 'Unclear'

yes_no_cols = [
    'understood_homepage',
    'explore_without_instructions',
    'difficulty_node_types',
    'wanted_back_reset',
    'visual_cues_helpful',
]

for col in yes_no_cols:
    df[col] = df[col].apply(normalize_yes_no)

# would_recommend has Yes / No / Maybe
def normalize_recommend(val):
    if pd.isna(val):
        return 'Unclear'
    v = str(val).strip().lower()
    if v.startswith('yes'):
        return 'Yes'
    if v.startswith('no'):
        return 'No'
    if v.startswith('maybe'):
        return 'Maybe'
    return 'Unclear'

df['would_recommend'] = df['would_recommend'].apply(normalize_recommend)

df[yes_no_cols + ['would_recommend']]

,understood_homepage,explore_without_instructions,difficulty_node_types,wanted_back_reset,visual_cues_helpful,would_recommend
0,Yes,Yes,No,Unclear,Yes,Yes
1,Yes,Yes,No,No,Yes,Yes
2,Yes,Unclear,No,Yes,Unclear,Maybe
3,Yes,No,No,No,Yes,Maybe
4,Yes,No,Yes,No,No,No
5,Yes,Yes,No,No,Yes,Yes
6,Yes,Yes,No,No,Yes,Yes


In [5]:
# Fill NaN text fields with empty string
text_cols = ['background', 'describe_experience', 'additional_feedback']
for col in text_cols:
    df[col] = df[col].fillna('')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   timestamp                     7 non-null      datetime64[ns]
 1   name                          7 non-null      object        
 2   age_group                     7 non-null      object        
 3   background                    7 non-null      object        
 4   experience_rating             7 non-null      int64         
 5   understood_homepage           7 non-null      object        
 6   explore_without_instructions  7 non-null      object        
 7   difficulty_node_types         7 non-null      object        
 8   wanted_back_reset             7 non-null      object        
 9   visual_cues_helpful           7 non-null      object        
 10  would_recommend               7 non-null      object        
 11  describe_experience           7 non-

In [6]:
# Export cleaned CSV
df.to_csv('cleaned_responses.csv', index=False)
print('Saved cleaned_responses.csv ✓')
df

Saved cleaned_responses.csv ✓


,timestamp,name,age_group,background,experience_rating,understood_homepage,explore_without_instructions,difficulty_node_types,wanted_back_reset,visual_cues_helpful,would_recommend,describe_experience,additional_feedback
0,2025-07-24 19:40:10,Steve Yang,Over 40,Engineer,7,Yes,Yes,No,Unclear,Yes,Yes,Not bad,No
1,2025-07-24 19:45:20,Wendy Ma,Over 40,Middle age career woman,8,Yes,Yes,No,No,Yes,Yes,Interesting tool to understand the knowledge o...,Probably it is helpful to have the inf. box mo...
2,2025-07-24 21:12:54,Katie ou,Over 40,"I’m a freelance food and health planner, helpi...",8,Yes,Unclear,No,Yes,Unclear,Maybe,,
3,2025-07-25 08:43:08,Hannah Yu,Over 40,,1,Yes,No,No,No,Yes,Maybe,,No
4,2025-07-25 09:18:44,John Guo,31-40,Biostatistician working on Parkinson's Disease...,3,Yes,No,Yes,No,No,No,"Great idea, has a lot of potential, needs a lo...",Steam engine overlapping with vaccination - tw...
5,2025-07-26 16:36:02,LIN LI,Over 40,Homemaker,9,Yes,Yes,No,No,Yes,Yes,Good way to understand the progress of technol...,No
6,2025-07-30 05:54:56,Eileen Zeng,Over 40,Research Scientist,10,Yes,Yes,No,No,Yes,Yes,技术树导航很直观，一目了然。,Not really
